In [3]:
pip install pyspark


  Using cached pyspark-4.1.2.tar.gz (455.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079558 sha256=7774d0f0de2a9c5b496293241694b9f70252adc41361bf68df10941660392fd7
  Stored in directory: c:\users\fleur\appdata\local\pip\cache\wheels\e6\9c\35\b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark

   ---------------------------------------- 0/2 [py4j]
   ---------------------------------------- 0/2 [py4j]
   ---------------------------------------- 0/2 [py4j]
   -------------

In [1]:
import os
import sys
import json
import glob
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, count

# ---------------------------------------------------------------------------
# GLOBAL PATHS CONFIGURATION & CRITICAL CONDAS/WINDOWS WORKER TIMEOUT FIX
# ---------------------------------------------------------------------------
# Force background processing worker threads to run on your exact active conda executable
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

# Map active Java 17 home paths dynamically inside the notebook registry
search_path = r"C:\Program Files\Eclipse Adoptium\jdk-17*"
found_folders = glob.glob(search_path)
if found_folders:
    os.environ["JAVA_HOME"] = found_folders[0]
    os.environ["PATH"] = os.path.join(found_folders[0], "bin") + os.pathsep + os.environ["PATH"]

# Initialize Local Standalone Spark Engine
spark = SparkSession.builder \
    .appName("AUCA_BigData_Final_Project") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.timeout", "800s") \
    .getOrCreate()

# Disable verbose background logs for clean deliverable visibility
spark.sparkContext.setLogLevel("ERROR")


def run_batch_processing_task():
    print("\n" + "="*55)
    print("  PYSPARK BATCH PROCESSING & CLEANING ")
    print("="*55)
    
    # 1. Ingest Structured File via Local JSON Parsing
    with open("transactions.json", "r") as f:
        raw_array_data = json.load(f)
    
   
    raw_df = spark.createDataFrame(pd.DataFrame(raw_array_data[:5000]))
    
    # 2. Data Cleaning & Normalization Transform
    clean_df = raw_df.filter(raw_df.transaction_id.isNotNull()) \
                     .filter(raw_df.user_id.isNotNull()) \
                     .filter(raw_df.status.isin(["completed", "shipped"]))
    
    # 3. Structural Transformation (Array Unwinding & Safe Field Extraction)
    exploded_df = clean_df.select(
        col("transaction_id"),
        explode(col("items")).alias("single_item")
    )
    
    flattened_df = exploded_df.select(
        col("transaction_id"),
        col("single_item.product_id").alias("product_id")
    ).filter(col("product_id").isNotNull())
    
    # 4. Matrix Generation via Self-Join
    df_p1 = flattened_df.alias("p1")
    df_p2 = flattened_df.alias("p2")
    
    pairs_df = df_p1.join(
        df_p2,
        (col("p1.transaction_id") == col("p2.transaction_id")) & 
        (col("p1.product_id") < col("p2.product_id"))  # Avoid duplicate mirror matching sets
    ).select(
        col("p1.product_id").alias("product_x"),
        col("p2.product_id").alias("product_y")
    )
    
    # 5. Frequency Computation
    recommendations = pairs_df.groupby("product_x", "product_y") \
                               .agg(count("*").alias("purchase_frequency")) \
                               .sort(col("purchase_frequency").desc())
    
    print("\n Top 15 Product Affinity Recommendations Matrix:")
    recommendations.show(15, truncate=False)
    
    # Return clean_df context to support Task 2 processing steps
    return clean_df


def run_spark_sql_analytics_task(clean_transactions_df):
    print("\n" + "="*55)
    print(" SPARK SQL NOTIONAL CROSS-DATABASE ANALYTICS ")
    print("="*55)
    
    # 1. Instantiate Demographic User Profiles (Notionally mapping MongoDB records)
    mock_mongodb_users = [
        {"user_id": "user_007129", "age_group": "25-34", "country": "Rwanda"},
        {"user_id": "user_000042", "age_group": "18-24", "country": "Kenya"},
        {"user_id": "user_001122", "age_group": "35-44", "country": "Uganda"}
    ]
    users_df = spark.createDataFrame(pd.DataFrame(mock_mongodb_users))
    
    # 2. Register Both Sources as Temp Relational Views inside Catalog
    clean_transactions_df.createOrReplaceTempView("hbase_transactions_view")
    users_df.createOrReplaceTempView("mongodb_users_view")
    
    # 3. Cross-Database Analytical SQL Query
    complex_cross_db_sql = """
        SELECT 
            u.country              AS geographic_region,
            u.age_group            AS customer_segment,
            COUNT(t.transaction_id) AS total_orders_processed,
            ROUND(SUM(t.total), 2)  AS total_gross_revenue
        FROM hbase_transactions_view t
        JOIN mongodb_users_view u ON t.user_id = u.user_id
        GROUP BY u.country, u.age_group
        ORDER BY total_gross_revenue DESC
    """
    
    print("\n Spark SQL Query Execution Result:")
    spark.sql(complex_cross_db_sql).show(truncate=False)


if __name__ == "__main__":
    # Execute full grading sequence tasks smoothly
    active_transactions = run_batch_processing_task()
    run_spark_sql_analytics_task(active_transactions)
    
    # Close session environment gracefully
    spark.stop()



  PYSPARK BATCH PROCESSING & CLEANING 

 Top 15 Product Affinity Recommendations Matrix:
+----------+----------+------------------+
|product_x |product_y |purchase_frequency|
+----------+----------+------------------+
|prod_02226|prod_03550|1                 |
|prod_02846|prod_04725|1                 |
|prod_00151|prod_04312|1                 |
|prod_00458|prod_03517|1                 |
|prod_01643|prod_04719|1                 |
|prod_02455|prod_04552|1                 |
|prod_02706|prod_04860|1                 |
|prod_00063|prod_03340|1                 |
|prod_00944|prod_03750|1                 |
|prod_03138|prod_03654|1                 |
|prod_03496|prod_04615|1                 |
|prod_00923|prod_00986|1                 |
|prod_01128|prod_03028|1                 |
|prod_02582|prod_04151|1                 |
|prod_02017|prod_03462|1                 |
+----------+----------+------------------+
only showing top 15 rows

 SPARK SQL NOTIONAL CROSS-DATABASE ANALYTICS 

 Spark SQL Query Exe